# Retrieval-Augmented Generation (RAG) with PDFs

This project demonstrates a complete RAG pipeline using Python, LangChain, FAISS, and OpenAI API.

Workflow:
1. Load 2-3 PDF documents
2. Chunk the extracted text
3. Generate embeddings
4. Store embeddings in FAISS
5. Retrieve relevant chunks via semantic search
6. Generate a grounded answer from retrieved context

In [6]:
%pip install -U langchain langchain-openai langchain-community faiss-cpu pypdf tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Dataset Setup

Add your own 2-3 PDF files into the `pdfs/` folder before running the notebook.

Example:
- pdfs/report_1.pdf
- pdfs/manual_2.pdf
- pdfs/policy_3.pdf

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_10064/3134731799.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Load Azure OpenAI settings from .env or environment
load_dotenv()

azure_api_key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "https://openai-api-management-gw.azure-api.net")
azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2023-05-15")
azure_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
azure_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")

if not azure_api_key:
    raise EnvironmentError("Missing API key. Set AZURE_OPENAI_API_KEY (or OPENAI_API_KEY) in .env or hardcode it in this cell.")

pdf_dir = Path("pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))

if len(pdf_files) < 2:
    raise ValueError("Please add at least 2 PDF files in the pdfs/ folder.")

print(f"Found {len(pdf_files)} PDF files:")
for p in pdf_files:
    print(f"- {p.name}")

print(f"Azure endpoint: {azure_endpoint}")
print(f"Embedding deployment: {azure_embedding_deployment}")
print(f"Chat deployment: {azure_chat_deployment}")

Found 3 PDF files:
- travel_agency_booking_policy.pdf
- travel_agency_customer_support_faq.pdf
- travel_agency_packages_2026.pdf
Azure endpoint: https://openai-api-management-gw.azure-api.net
Embedding deployment: text-embedding-ada-002
Chat deployment: gpt-4o-mini


In [3]:
# Load pages from all PDFs
documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    for page in pages:
        page.metadata["source_file"] = pdf_path.name
    documents.extend(pages)

print(f"Loaded {len(documents)} pages from {len(pdf_files)} files.")

Loaded 3 pages from 3 files.


In [4]:
# Chunk documents for retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [5]:
# Build embeddings and FAISS vector database
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=azure_api_version,
    deployment=azure_embedding_deployment
)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print("FAISS index built and saved to faiss_index/")

FAISS index built and saved to faiss_index/


In [6]:
# Semantic retrieval
query = "What are the most important recommendations across these documents?"
k = 3
retrieved = vectorstore.similarity_search_with_score(query, k=k)

print("Top retrieved chunks:")
for i, (doc, score) in enumerate(retrieved, start=1):
    print(f"\nResult {i} | score={score:.4f}")
    print(f"Source: {doc.metadata.get('source_file', 'unknown')} | Page: {doc.metadata.get('page', 'n/a')}")
    print(doc.page_content[:400].replace("\n", " ") + "...")

Top retrieved chunks:

Result 1 | score=0.5710
Source: travel_agency_customer_support_faq.pdf | Page: 0
Skyline Travels - Customer Support FAQ Before You Travel - Q: When will I get my final itinerary? - A: Usually 5-7 days before departure via email. - Q: Can I request vegetarian meals? - A: Yes, meal preferences can be submitted 10 days before departure. - Q: Do you assist with visas? - A: Yes, we provide checklist guidance and appointment support. During Your Trip - 24/7 helpline: +1-800-555-0199...

Result 2 | score=0.5808
Source: travel_agency_packages_2026.pdf | Page: 0
Skyline Travels - Holiday Packages 2026 Europe Explorer (10 Days) - Cities: Paris, Zurich, Venice, Rome - Price: USD 2,450 per person (double occupancy) - Includes: 4-star hotels, daily breakfast, airport transfers - Optional add-on: Swiss scenic rail pass for USD 180 - Best season: April to June, September to October Southeast Asia Discovery (8 Days) - Cities: Bangkok, Siem Reap, Ho Chi Minh City...

Result 3 | s

In [7]:
# RAG answer generation
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=azure_api_version,
    deployment_name=azure_chat_deployment,
    temperature=0,
)
retrieved_docs = [doc for doc, _ in retrieved]
context = "\n\n".join(
    [
        f"Source: {d.metadata.get('source_file', 'unknown')} | Page: {d.metadata.get('page', 'n/a')}\n{d.page_content}"
        for d in retrieved_docs
    ]
)

prompt = f"""Use only the context below to answer the question.
If the answer is not in the context, say: Not found in the provided documents.

Question: {query}

Context:
{context}
"""

answer = llm.invoke(prompt).content
print("Answer:\n")
print(answer)

Answer:

The most important recommendations across the provided documents are:

1. **Before You Travel**:
   - Final itinerary is sent 5-7 days before departure.
   - Submit meal preferences (e.g., vegetarian) 10 days before departure.
   - Assistance with visas is available.

2. **During Your Trip**:
   - Use the 24/7 helpline for urgent assistance.
   - Contact local ground partner as per travel voucher.
   - Follow emergency protocol: contact helpline first, then nearest embassy.
   - Expect a response within 20 minutes for missed transfer support.

3. **After Your Trip**:
   - Refund claims processed within 10 to 15 business days.
   - Service feedback forms sent within 48 hours of return.
   - Loyalty members earn points for bookings; repeat travelers get discounts.

4. **Booking and Cancellation Policy**:
   - A 30% deposit is required to confirm bookings; full payment is due 21 days before departure.
   - Travel insurance is recommended.
   - Cancellation policies vary based on 

In [8]:
# Save sample output for LMS submission
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
sample_output_path = output_dir / "sample_output.txt"

with open(sample_output_path, "w", encoding="utf-8") as f:
    f.write(f"Query: {query}\n\n")
    f.write("Top retrieved chunks:\n")
    for i, (doc, score) in enumerate(retrieved, start=1):
        f.write(f"\n[{i}] score={score:.4f}\n")
        f.write(
            f"source={doc.metadata.get('source_file', 'unknown')} page={doc.metadata.get('page', 'n/a')}\n"
        )
        f.write(doc.page_content[:600] + "\n")

    f.write("\nGenerated answer:\n")
    f.write(answer)

print(f"Sample output saved to: {sample_output_path.resolve()}")

Sample output saved to: /home/asima.khalid557/rag_pipeline/outputs/sample_output.txt


## Short Analysis Report

- Chunking strategy: `RecursiveCharacterTextSplitter` with `chunk_size=1000` and `chunk_overlap=150` to preserve context across boundaries.
- Embedding model: `text-embedding-3-small` chosen for a good quality/cost trade-off.
- Vector database: FAISS used for efficient local similarity search.
- Retrieval observations: Specific questions generally return better and more grounded chunks than broad questions.
- Limitations: Scanned PDFs and complex tables may reduce extraction quality and retrieval precision.
- Next improvement: Add metadata filtering and reranking to improve top-k relevance.

## Submission Checklist

Zip and submit this project folder with:
- Completed notebook (`main.ipynb`)
- 2-3 PDF files in `pdfs/`
- Sample outputs (`outputs/sample_output.txt` and notebook outputs)
- Short analysis report (`analysis_report.md` or notebook section above)